# Notebook 16: Federated Learning Attacks

## Why This Matters

Federated learning's promise — privacy through data locality — rests on an optimistic assumption: that all participating clients are honest. In production FL deployments, this assumption is often false. A federated system might include thousands of clients spanning consumer devices, hospitals, banks, or phones in adversarial nations. Any one of them could be compromised, malicious, or simply misconfigured.

FL attacks are structurally different from centralized ML attacks. In centralized settings, the adversary is outside the training process — they can only query the final model. In FL, malicious clients are *inside* the training process. They upload model updates that the server aggregates alongside honest clients. This insider position enables attack classes that have no centralized equivalent.

The attacks in this notebook are well-documented in the academic literature and have practical implications for any real FL deployment: healthcare FL consortia, cross-silo financial ML, and on-device federated training at scale.

## Background

### Byzantine Attacks

In distributed systems, a Byzantine fault is a failure where a component can send arbitrary, conflicting information to different parts of the system. FL inherits this problem: a Byzantine client can upload any gradient or weight update — including crafted ones designed to corrupt the global model.

The simplest Byzantine attack is *gradient scaling*: multiply your gradient by a large constant to overwhelm honest clients. More sophisticated attacks compute gradients designed to maximize a backdoor objective while remaining statistically indistinguishable from honest gradients (Bhagoji et al., 2019).

### Backdoor Attacks in FL

Bagdasaryan et al. (2020) showed that a single malicious client can inject a persistent backdoor into the global model, even with a small fraction of clients. The attack works by:
1. Training locally on both honest and poisoned data
2. Scaling the malicious update by `1/fraction_of_malicious_clients` — this amplifies the poisoned update to compensate for the honest clients' dampening effect
3. The global model learns the backdoor behavior but maintains clean accuracy (making detection harder)

This "scaling attack" is particularly dangerous because the server cannot distinguish the malicious update from an honest one by inspecting the weights alone.

### Gradient Inversion / Deep Leakage

Zhu et al.'s 2019 paper *"Deep Leakage from Gradients"* (NeurIPS, 300+ citations) showed that the gradients uploaded by an FL client can be *inverted* to reconstruct the original training data. Given a gradient vector `∇L(x, y)`, an adversary can recover `(x, y)` by optimizing a dummy input to match the observed gradient:

```
x* = argmin_{x'} ||∇L(x', y') - ∇L(x, y)||²
```

This works because the gradient of a neural network encodes information about the input — the backpropagation signal is not a one-way hash. The attack works best for small batches (batch size 1-4) and degrades for larger batches, but even batch gradients can partially reveal sensitive information.

### Byzantine-Robust Aggregation

Standard FedAvg averages client updates, which makes it highly vulnerable — a single malicious client with a 10x scaled update can dominate the average. Several robust aggregation methods have been proposed:

- **Krum** (Blanchard et al., 2017): Select the single client whose update is closest to its k nearest neighbors. Ignores outliers entirely.
- **Trimmed Mean**: Sort gradients by each coordinate, trim the highest and lowest fraction, average the rest.
- **Median**: Take coordinate-wise median instead of mean. O(n) per coordinate, O(n) breakdown point.
- **FLTrust** (Cao et al., 2020): Server maintains a small trusted dataset; scores client updates by cosine similarity to server's own gradient.

None of these methods are perfect — they all have failure modes against adaptive adversaries who craft updates to pass the robustness check.

## Structure of This Notebook

1. Byzantine attacks: gradient scaling and random noise injection
2. Backdoor injection via the scaling attack
3. Gradient inversion — reconstructing training images from gradients
4. Robust aggregation: Krum, trimmed mean, coordinate-wise median
5. Attack vs. defense: which attacks survive which defenses?

## What You'll Learn

- Why FL's aggregation step is the attack surface
- How gradient scaling amplifies a malicious client's influence
- How gradient inversion reconstructs training data from uploaded updates
- How Krum, trimmed mean, and median aggregation resist Byzantine attacks
- Why adaptive attackers can often bypass robustness defenses

In [ ]:
%pip install torch torchvision numpy matplotlib --quiet

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import copy

device = (
    torch.device('mps') if torch.backends.mps.is_available()
    else torch.device('cuda') if torch.cuda.is_available()
    else torch.device('cpu')
)
print(f'Device: {device}')
torch.manual_seed(42)
np.random.seed(42)

## 1. FL Environment Setup

In [ ]:
class MnistCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.drop = nn.Dropout(0.25)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 64 * 7 * 7)
        x = F.relu(self.fc1(x))
        x = self.drop(x)
        return self.fc2(x)


transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
train_dataset = torchvision.datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.MNIST('./data', train=False, download=True, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=512, shuffle=False)

train_labels = np.array([train_dataset[i][1] for i in range(len(train_dataset))])

# Simple IID partition for 20 clients
N_CLIENTS = 20
all_idx = np.random.permutation(len(train_dataset))
client_indices = [list(split) for split in np.array_split(all_idx, N_CLIENTS)]

print(f'Setup: {N_CLIENTS} clients, ~{len(client_indices[0])} samples each')


def get_weights(model):
    return copy.deepcopy(model.state_dict())

def set_weights(model, weights):
    model.load_state_dict(weights)

def evaluate(weights, test_loader):
    model = MnistCNN().to(device)
    set_weights(model, weights)
    model.eval()
    correct = 0
    with torch.no_grad():
        for x, y in test_loader:
            correct += (model(x.to(device)).argmax(1) == y.to(device)).sum().item()
    return correct / len(test_loader.dataset)

def weights_to_vec(weights):
    """Flatten all weight tensors into a single vector."""
    return torch.cat([v.flatten().float() for v in weights.values()])

def vec_to_weights(vec, template_weights):
    """Unflatten a vector back to the weight dict structure."""
    result = {}
    offset = 0
    for key, val in template_weights.items():
        n = val.numel()
        result[key] = vec[offset:offset+n].reshape(val.shape).to(val.dtype)
        offset += n
    return result


def honest_client_update(global_weights, data_idx, dataset, local_epochs=3, lr=0.01):
    """Standard honest client: local SGD, uploads real update."""
    model = MnistCNN().to(device)
    set_weights(model, global_weights)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    loader = DataLoader(Subset(dataset, data_idx), batch_size=32, shuffle=True)

    model.train()
    for _ in range(local_epochs):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            F.cross_entropy(model(x), y).backward()
            opt.step()

    return get_weights(model)

## 2. Byzantine Attacks — Gradient Scaling and Random Noise

In [ ]:
def byzantine_scale_attack(global_weights, honest_update, scale_factor=10.0):
    """
    Scale attack: take the *opposite* of the honest gradient direction,
    amplified by scale_factor.
    Goal: push global model in the wrong direction.
    """
    malicious = {}
    for key in global_weights:
        delta = honest_update[key] - global_weights[key]
        malicious[key] = global_weights[key] - scale_factor * delta  # reverse + amplify
    return malicious


def byzantine_random_attack(global_weights, noise_scale=1.0):
    """Random noise attack: upload random weights (extremely simple Byzantine fault)."""
    malicious = {}
    for key, val in global_weights.items():
        malicious[key] = torch.randn_like(val.float()) * noise_scale
    return malicious


def standard_aggregate(client_updates, client_sizes):
    """Standard FedAvg: weighted mean."""
    total = sum(client_sizes)
    result = {}
    for key in client_updates[0]:
        result[key] = sum(u[key].float() * (s / total)
                          for u, s in zip(client_updates, client_sizes))
    return result


def run_fl_with_attacks(
    client_indices,
    dataset,
    test_loader,
    n_rounds: int,
    clients_per_round: int,
    n_byzantine: int,
    attack_type: str,       # 'scale', 'random', 'none'
    aggregation: str,       # 'mean', 'krum', 'trimmed_mean', 'median'
    local_epochs: int = 3,
    lr: float = 0.01,
    scale_factor: float = 10.0,
):
    global_model = MnistCNN().to(device)
    global_weights = get_weights(global_model)
    n_clients = len(client_indices)
    history = []

    for r in range(n_rounds):
        selected = np.random.choice(n_clients, clients_per_round, replace=False)
        updates = []
        sizes = []

        for i, c in enumerate(selected):
            is_byzantine = (i < n_byzantine)
            honest_w = honest_client_update(global_weights, client_indices[c], dataset,
                                            local_epochs=local_epochs, lr=lr)

            if is_byzantine and attack_type == 'scale':
                w = byzantine_scale_attack(global_weights, honest_w, scale_factor)
            elif is_byzantine and attack_type == 'random':
                w = byzantine_random_attack(global_weights)
            else:
                w = honest_w

            updates.append(w)
            sizes.append(len(client_indices[c]))

        # Aggregate
        global_weights = aggregate(updates, sizes, aggregation)

        acc = evaluate(global_weights, test_loader)
        history.append(acc)

    return history


def aggregate(updates, sizes, method):
    """Dispatch to aggregation method."""
    if method == 'mean':
        return standard_aggregate(updates, sizes)
    elif method == 'krum':
        return krum_aggregate(updates)
    elif method == 'trimmed_mean':
        return trimmed_mean_aggregate(updates, trim_fraction=0.2)
    elif method == 'median':
        return median_aggregate(updates)
    else:
        raise ValueError(f'Unknown aggregation: {method}')


print('Byzantine attack framework implemented.')

In [ ]:
# Robust aggregation implementations

def krum_aggregate(updates, n_byzantine=None):
    """
    Krum aggregation (Blanchard et al., 2017):
    For each client, compute sum of squared distances to its k-nearest neighbors.
    Select the client with the minimum score (most "central" update).
    Robust to up to floor((n-2)/2) Byzantine clients.
    """
    n = len(updates)
    if n_byzantine is None:
        n_byzantine = max(0, n // 4)  # assume up to 25% Byzantine
    k = n - n_byzantine - 2

    # Flatten all updates to vectors
    vecs = torch.stack([weights_to_vec(u) for u in updates])

    # Pairwise distances
    scores = torch.zeros(n)
    for i in range(n):
        dists = torch.sum((vecs - vecs[i]) ** 2, dim=1)
        dists[i] = float('inf')  # exclude self
        knn_dists = torch.topk(dists, k, largest=False).values
        scores[i] = knn_dists.sum()

    best_idx = scores.argmin().item()
    return updates[best_idx]


def trimmed_mean_aggregate(updates, trim_fraction=0.2):
    """
    Coordinate-wise trimmed mean:
    For each parameter, sort the values from all clients,
    trim the top and bottom trim_fraction, then average.
    """
    result = {}
    n = len(updates)
    n_trim = max(1, int(n * trim_fraction))

    for key in updates[0]:
        stacked = torch.stack([u[key].float() for u in updates], dim=0)
        sorted_vals, _ = torch.sort(stacked, dim=0)
        trimmed = sorted_vals[n_trim:n - n_trim]
        result[key] = trimmed.mean(dim=0)

    return result


def median_aggregate(updates):
    """Coordinate-wise median aggregation. Breakdown point = 50%."""
    result = {}
    for key in updates[0]:
        stacked = torch.stack([u[key].float() for u in updates], dim=0)
        result[key] = stacked.median(dim=0).values
    return result


print('Robust aggregation methods (Krum, Trimmed Mean, Median) implemented.')

In [ ]:
N_ROUNDS = 15
CLIENTS_PER_ROUND = 10
N_BYZANTINE = 3  # 30% Byzantine

print('Running attack experiments (FedAvg mean aggregation)...')

print('\n[1] No attack baseline')
hist_clean = run_fl_with_attacks(
    client_indices, train_dataset, test_loader,
    n_rounds=N_ROUNDS, clients_per_round=CLIENTS_PER_ROUND,
    n_byzantine=0, attack_type='none', aggregation='mean'
)

print('\n[2] Scale attack (mean aggregation)')
hist_scale = run_fl_with_attacks(
    client_indices, train_dataset, test_loader,
    n_rounds=N_ROUNDS, clients_per_round=CLIENTS_PER_ROUND,
    n_byzantine=N_BYZANTINE, attack_type='scale', aggregation='mean',
    scale_factor=10.0
)

print('\n[3] Random noise attack (mean aggregation)')
hist_random = run_fl_with_attacks(
    client_indices, train_dataset, test_loader,
    n_rounds=N_ROUNDS, clients_per_round=CLIENTS_PER_ROUND,
    n_byzantine=N_BYZANTINE, attack_type='random', aggregation='mean'
)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(range(1, N_ROUNDS+1), hist_clean, 'b-o', markersize=4, label='No attack')
ax.plot(range(1, N_ROUNDS+1), hist_scale, 'r-s', markersize=4,
        label=f'Scale attack ({N_BYZANTINE}/{CLIENTS_PER_ROUND} Byzantine, ×10)')
ax.plot(range(1, N_ROUNDS+1), hist_random, 'orange', marker='^', markersize=4,
        label=f'Random noise attack ({N_BYZANTINE}/{CLIENTS_PER_ROUND} Byzantine)')
ax.set_xlabel('Communication Rounds')
ax.set_ylabel('Global Model Test Accuracy')
ax.set_title('Byzantine Attacks on FedAvg (Mean Aggregation)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Robust Aggregation vs. Attacks

In [ ]:
print('Running robust aggregation experiments against scale attack...')

print('\n[1] Krum aggregation')
hist_krum = run_fl_with_attacks(
    client_indices, train_dataset, test_loader,
    n_rounds=N_ROUNDS, clients_per_round=CLIENTS_PER_ROUND,
    n_byzantine=N_BYZANTINE, attack_type='scale', aggregation='krum'
)

print('\n[2] Trimmed mean aggregation')
hist_trimmed = run_fl_with_attacks(
    client_indices, train_dataset, test_loader,
    n_rounds=N_ROUNDS, clients_per_round=CLIENTS_PER_ROUND,
    n_byzantine=N_BYZANTINE, attack_type='scale', aggregation='trimmed_mean'
)

print('\n[3] Coordinate-wise median')
hist_median = run_fl_with_attacks(
    client_indices, train_dataset, test_loader,
    n_rounds=N_ROUNDS, clients_per_round=CLIENTS_PER_ROUND,
    n_byzantine=N_BYZANTINE, attack_type='scale', aggregation='median'
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(1, N_ROUNDS+1), hist_clean, 'b-o', markersize=4, label='No attack (mean)')
ax.plot(range(1, N_ROUNDS+1), hist_scale, 'r-s', markersize=4, label='Scale attack (mean) — VULNERABLE')
ax.plot(range(1, N_ROUNDS+1), hist_krum, 'g-^', markersize=4, label='Scale attack (Krum)')
ax.plot(range(1, N_ROUNDS+1), hist_trimmed, 'purple', marker='D', markersize=4,
        label='Scale attack (Trimmed Mean)')
ax.plot(range(1, N_ROUNDS+1), hist_median, 'orange', marker='v', markersize=4,
        label='Scale attack (Median)')
ax.set_xlabel('Communication Rounds')
ax.set_ylabel('Test Accuracy')
ax.set_title('Robust Aggregation Defenses vs. Scale Attack (30% Byzantine)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'\nFinal accuracies:')
print(f'  Clean (mean):     {hist_clean[-1]:.4f}')
print(f'  Scale (mean):     {hist_scale[-1]:.4f} (attacked)')
print(f'  Scale (Krum):     {hist_krum[-1]:.4f}')
print(f'  Scale (Trimmed):  {hist_trimmed[-1]:.4f}')
print(f'  Scale (Median):   {hist_median[-1]:.4f}')

## 4. Gradient Inversion — Reconstructing Training Data from Updates

In [ ]:
def gradient_inversion_attack(
    model,
    true_gradient,  # dict of param_name -> gradient tensor
    n_images: int = 1,
    n_steps: int = 300,
    lr: float = 0.1,
    tv_weight: float = 1e-4,
):
    """
    Deep Leakage from Gradients (Zhu et al., 2019).

    The server (adversary) observed a client's gradient update.
    Goal: recover the original training image by finding x* that
    produces the same gradient.

    Optimize: x*, y* = argmin ||∇L(f(x*), y*) - g||²
    where g is the observed gradient.

    Total variation regularization encourages smooth (natural-looking) reconstructions.
    """
    # Initialize random dummy input and label
    dummy_x = torch.randn(n_images, 1, 28, 28, device=device, requires_grad=True)
    dummy_y = torch.zeros(n_images, 10, device=device, requires_grad=True)

    optimizer = torch.optim.Adam([dummy_x, dummy_y], lr=lr)

    # Flatten true gradient to vector for comparison
    true_grad_vec = torch.cat([g.flatten() for g in true_gradient.values()])

    losses = []
    for step in range(n_steps):
        optimizer.zero_grad()

        # Compute gradient for dummy input
        model.zero_grad()
        dummy_pred = model(dummy_x)
        dummy_loss = F.cross_entropy(dummy_pred, F.softmax(dummy_y, dim=1))
        dummy_grads = torch.autograd.grad(dummy_loss, model.parameters(),
                                          create_graph=True)
        dummy_grad_vec = torch.cat([g.flatten() for g in dummy_grads])

        # Gradient matching loss
        grad_loss = ((dummy_grad_vec - true_grad_vec) ** 2).sum()

        # Total variation regularization (smoothness prior)
        tv_loss = tv_weight * (
            ((dummy_x[:, :, :-1, :] - dummy_x[:, :, 1:, :]) ** 2).sum() +
            ((dummy_x[:, :, :, :-1] - dummy_x[:, :, :, 1:]) ** 2).sum()
        )

        total_loss = grad_loss + tv_loss
        total_loss.backward()
        optimizer.step()
        losses.append(grad_loss.item())

    return dummy_x.detach(), losses


def compute_true_gradient(model, x, y):
    """Compute the gradient that a client would upload for input (x, y)."""
    model.zero_grad()
    loss = F.cross_entropy(model(x.to(device)), y.to(device))
    loss.backward()
    return {name: p.grad.clone() for name, p in model.named_parameters() if p.grad is not None}


# Victim model (partially trained)
victim = MnistCNN().to(device)
global_weights = get_weights(victim)

# Pick a real training image
sample_idx = 42
x_true, y_true = train_dataset[sample_idx]
x_true = x_true.unsqueeze(0)
y_true_tensor = torch.tensor([y_true])

print(f'Target: digit {y_true}')
print('Computing true gradient from client...')
true_grad = compute_true_gradient(victim, x_true, y_true_tensor)

print('Running gradient inversion attack (300 steps)...')
reconstructed, losses = gradient_inversion_attack(
    victim, true_grad, n_images=1, n_steps=300, lr=0.05, tv_weight=1e-4
)

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(10, 3))

# Normalize for display
x_display = x_true.squeeze().cpu().numpy()
r_display = reconstructed.squeeze().cpu().numpy()
r_display = (r_display - r_display.min()) / (r_display.max() - r_display.min() + 1e-8)

axes[0].imshow(x_display, cmap='gray')
axes[0].set_title(f'Original (label={y_true})')
axes[0].axis('off')

axes[1].imshow(r_display, cmap='gray')
axes[1].set_title('Reconstructed from Gradient')
axes[1].axis('off')

axes[2].plot(losses)
axes[2].set_xlabel('Optimization Step')
axes[2].set_ylabel('Gradient Matching Loss')
axes[2].set_title('Inversion Convergence')
axes[2].set_yscale('log')
axes[2].grid(True, alpha=0.3)

plt.suptitle('Gradient Inversion Attack (Zhu et al., 2019)', fontsize=12)
plt.tight_layout()
plt.show()

# Compute reconstruction error
mse = ((x_true.cpu() - reconstructed.cpu())**2).mean().item()
print(f'Reconstruction MSE: {mse:.6f}')
print('Lower MSE = better reconstruction. Works best at batch size 1.')

## 5. FL Backdoor via Scaling Attack

In [ ]:
TRIGGER_PATCH_SIZE = 4
BACKDOOR_TARGET = 0

def apply_trigger(x):
    """Add a white patch in bottom-right corner."""
    x = x.clone()
    x[..., -TRIGGER_PATCH_SIZE:, -TRIGGER_PATCH_SIZE:] = 1.0
    return x


def malicious_backdoor_update(
    global_weights,
    data_idx,
    dataset,
    n_honest_clients: int,  # for scaling calculation
    local_epochs: int = 5,
    lr: float = 0.01,
    poison_rate: float = 0.3,
):
    """
    Backdoor + scaling attack (Bagdasaryan et al., 2020).

    1. Train on mixed data: honest samples + triggered (poisoned) samples
    2. Scale the weight delta by n_honest_clients to survive averaging
    """
    model = MnistCNN().to(device)
    set_weights(model, global_weights)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)

    subset = Subset(dataset, data_idx)
    loader = DataLoader(subset, batch_size=32, shuffle=True)

    model.train()
    for _ in range(local_epochs):
        for x, y in loader:
            # Create poisoned batch
            n_poison = int(len(x) * poison_rate)
            x_poison = apply_trigger(x[:n_poison].clone())
            y_poison = torch.full((n_poison,), BACKDOOR_TARGET, dtype=torch.long)

            x_all = torch.cat([x, x_poison]).to(device)
            y_all = torch.cat([y, y_poison]).to(device)

            opt.zero_grad()
            F.cross_entropy(model(x_all), y_all).backward()
            opt.step()

    # Scale attack: amplify delta to survive FedAvg averaging
    malicious_weights = get_weights(model)
    scaled = {}
    for key in global_weights:
        delta = malicious_weights[key].float() - global_weights[key].float()
        scaled[key] = global_weights[key].float() + n_honest_clients * delta

    return scaled


def evaluate_backdoor(weights, test_loader):
    """Evaluate both clean accuracy and backdoor attack success rate."""
    model = MnistCNN().to(device)
    set_weights(model, weights)
    model.eval()

    clean_correct = 0
    backdoor_triggered = 0
    total = 0

    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            # Clean accuracy
            clean_correct += (model(x).argmax(1) == y).sum().item()
            # Backdoor ASR: triggered inputs classified as target
            x_triggered = apply_trigger(x).to(device)
            backdoor_triggered += (model(x_triggered).argmax(1) == BACKDOOR_TARGET).sum().item()
            total += len(y)

    return clean_correct / total, backdoor_triggered / total


# Run FL with one malicious client doing scaling backdoor attack
global_w = get_weights(MnistCNN().to(device))

clean_accs = []
backdoor_asrs = []
N_ROUNDS_BD = 15
CLIENTS_PER_ROUND_BD = 8
N_HONEST = CLIENTS_PER_ROUND_BD - 1

print('Running FL with backdoor scaling attack (1/8 clients malicious)...')
for r in range(N_ROUNDS_BD):
    selected = np.random.choice(len(client_indices), CLIENTS_PER_ROUND_BD, replace=False)
    updates = []
    sizes = []

    for i, c in enumerate(selected):
        if i == 0:  # malicious client
            w = malicious_backdoor_update(global_w, client_indices[c], train_dataset,
                                          n_honest_clients=N_HONEST)
        else:
            w = honest_client_update(global_w, client_indices[c], train_dataset)
        updates.append(w)
        sizes.append(len(client_indices[c]))

    global_w = standard_aggregate(updates, sizes)
    clean_acc, asr = evaluate_backdoor(global_w, test_loader)
    clean_accs.append(clean_acc)
    backdoor_asrs.append(asr)

    if (r + 1) % 5 == 0:
        print(f'Round {r+1}: clean_acc={clean_acc:.4f}, backdoor_ASR={asr:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(range(1, N_ROUNDS_BD+1), clean_accs, 'b-o', markersize=4)
axes[0].set_title('Clean Accuracy')
axes[0].set_xlabel('Rounds')
axes[0].set_ylabel('Accuracy')
axes[0].grid(True, alpha=0.3)

axes[1].plot(range(1, N_ROUNDS_BD+1), backdoor_asrs, 'r-o', markersize=4)
axes[1].set_title('Backdoor Attack Success Rate (ASR)')
axes[1].set_xlabel('Rounds')
axes[1].set_ylabel('ASR (fraction triggered→target)')
axes[1].grid(True, alpha=0.3)

plt.suptitle('FL Backdoor Scaling Attack (1 malicious / 8 clients)', fontsize=12)
plt.tight_layout()
plt.show()

print(f'Final: clean_acc={clean_accs[-1]:.4f}, ASR={backdoor_asrs[-1]:.4f}')
print('The scaling attack injects a backdoor while maintaining clean accuracy.')

## Summary

### What We Built

| Attack/Defense | Implementation | Key Mechanism |
|----------------|---------------|---------------|
| Scale attack | `byzantine_scale_attack()` | Reverses + amplifies gradient direction |
| Random noise | `byzantine_random_attack()` | Uploads random weight tensor |
| Backdoor + scaling | `malicious_backdoor_update()` | Poisons locally, scales delta by n_clients |
| Gradient inversion | `gradient_inversion_attack()` | Optimizes dummy input to match observed gradient |
| Krum aggregation | `krum_aggregate()` | Selects most "central" update by k-NN distance |
| Trimmed mean | `trimmed_mean_aggregate()` | Coordinate-wise trim top/bottom 20% |
| Coordinate median | `median_aggregate()` | 50% breakdown point per coordinate |

### Key Takeaways

- **Standard FedAvg (mean) is highly vulnerable.** A single client with a 10x scaled update in the opposite direction can devastate the global model.
- **The scaling backdoor attack is particularly dangerous** because it maintains clean accuracy while injecting the backdoor — the server cannot detect it by evaluating accuracy alone.
- **Gradient inversion is a real privacy threat.** For batch size 1-4, the server can reconstruct training images from uploaded gradients. This is why FL alone does not guarantee data privacy without DP.
- **Robust aggregation helps but doesn't solve everything.** Krum, trimmed mean, and median all resist naive scale attacks, but adaptive adversaries who craft updates to pass the robustness check can still succeed.
- **The complete FL defense stack** combines: robust aggregation + DP on updates + secure aggregation (so the server never sees individual updates) + anomaly detection.

### The Fundamental Tension

FL was motivated by privacy (raw data stays local), but the uploaded gradients themselves leak information (gradient inversion). DP on the uploaded gradients addresses leakage but costs accuracy. Robust aggregation addresses Byzantine attacks but may reject legitimate updates from non-IID clients. These defenses interact in complex ways, and no production FL system has fully solved the tradeoff.

Next: Notebook 17 covers ML model watermarking — embedding ownership proofs in model weights or outputs that survive fine-tuning and can be verified in court as forensic evidence of model theft.